# 06 Internal Statistical Diagnostics

This notebook is internal-facing. It uses synthetic evaluation labels to examine uncertainty, subgroup behavior, expected-pay calibration, exposure calibration, robustness, and score sensitivity without changing analyst-safe queue outputs.

In [1]:
import polars as pl
from common.execution import notebook_fast_mode
from common.plots import (
    LetsPlot,
    aes,
    geom_errorbar,
    geom_point,
    geom_tile,
    ggplot,
    ggtitle,
    theme_minimal,
)

from payroll_anomaly_ranking.config import PayrollConfig
from payroll_anomaly_ranking.data import scenario_sanity_summary
from payroll_anomaly_ranking.diagnostics import (
    calibration_plot_inputs,
    exposure_calibration,
    pairwise_component_superiority,
    perturbation_sensitivity,
    review_budget_interval_summary,
    robustness_summary,
    run_diagnostic_comparison_units,
    subgroup_diagnostics,
    top_subgroup_diagnostics,
)
from payroll_anomaly_ranking.models import score_payroll
from payroll_anomaly_ranking.pipeline import PipelineIncludeConfig, run_pipeline
from payroll_anomaly_ranking.scenarios import diagnostic_scenario_presets

LetsPlot.setup_html()

In [2]:
config = PayrollConfig(employee_count=220, pay_periods=14, review_budgets=(10, 25))
DIAGNOSTIC_SCENARIOS = (
    "baseline",
    "rule-friendly",
    "statistical-friendly",
    "ml-friendly",
    "exposure-heavy",
    "subgroup-drift",
    "calendar-drift",
    "queue-stress",
)
DIAGNOSTIC_SEEDS = (42, 43, 44)
INTERVAL_SAMPLES = 75
FAST_MODE_SCENARIOS = ("baseline", "subgroup-drift", "queue-stress")
FAST_MODE_SEEDS = (42,)
FAST_MODE_SAMPLE_COUNT = 25
FAST_MODE_NOTE = "Dense defaults: 8 scenarios, 3 seeds, 220 employees, 14 pay periods, samples=75. Fast mode: reduce to FAST_MODE_SCENARIOS, FAST_MODE_SEEDS, or FAST_MODE_SAMPLE_COUNT."
NOTEBOOK_FAST = notebook_fast_mode()
active_scenarios = FAST_MODE_SCENARIOS if NOTEBOOK_FAST else DIAGNOSTIC_SCENARIOS
active_seeds = FAST_MODE_SEEDS if NOTEBOOK_FAST else DIAGNOSTIC_SEEDS
active_interval_samples = FAST_MODE_SAMPLE_COUNT if NOTEBOOK_FAST else INTERVAL_SAMPLES
active_pipeline_include = (
    PipelineIncludeConfig.scored_only()
    if NOTEBOOK_FAST
    else PipelineIncludeConfig.all()
)
scenarios = diagnostic_scenario_presets(active_scenarios)
results = run_pipeline(
    config,
    scenario=scenarios["subgroup-drift"],
    include=active_pipeline_include,
)
scored = results.scored

In [3]:
sanity = pl.concat(
    [
        scenario_sanity_summary(
            run_pipeline(
                config,
                scenario=scenario,
                include=active_pipeline_include,
            ).scored,
            scenario=name,
        )
        for name, scenario in scenarios.items()
    ],
)
sanity

scenario,row_count,anomaly_count,anomaly_dollars,score_p50,score_p90,category_mix,max_subgroup_period_anomaly_share,zero_threshold_candidates,sparse_condition
str,i64,i64,f64,f64,f64,str,f64,str,str
"""baseline""",15084,167,53153.63,0.132019,0.279853,"""normal=14917;overtime_double_s…",0.041916,"""none""","""none"""
"""rule-friendly""",14331,130,2759.23,0.131901,0.281954,"""normal=14201;unsupported_shift…",0.046154,"""none""","""none"""
"""statistical-friendly""",15407,140,45708.04,0.139496,0.299292,"""normal=15267;overtime_double_s…",0.035714,"""none""","""none"""
"""ml-friendly""",12660,130,2802.13,0.133184,0.274311,"""normal=12530;unsupported_shift…",0.046154,"""none""","""none"""
"""exposure-heavy""",14627,140,43604.99,0.129724,0.270625,"""normal=14487;overtime_double_s…",0.035714,"""none""","""none"""
"""subgroup-drift""",14627,140,43604.99,0.129724,0.270625,"""normal=14487;overtime_double_s…",0.035714,"""none""","""none"""
"""calendar-drift""",14593,130,2776.15,0.129802,0.288308,"""normal=14463;unsupported_shift…",0.038462,"""none""","""none"""
"""queue-stress""",13311,140,43579.77,0.136634,0.281384,"""normal=13171;overtime_double_s…",0.035714,"""none""","""none"""


## Review Budget Intervals And Multi-Regime Component Superiority

Diagnostic question: which ranking signal wins when the synthetic world changes? These scenario regimes are internal stress tests, not estimates of real payroll frequencies.

In [4]:
intervals = review_budget_interval_summary(
    scored,
    k=10,
    samples=active_interval_samples,
    seed=config.seed,
)
unit_metrics = run_diagnostic_comparison_units(
    config,
    scenarios=scenarios,
    seeds=active_seeds,
    k=10,
)
superiority = pairwise_component_superiority(unit_metrics, metric="precision_at_k")
intervals

metric,k,mean,lower_95,upper_95,method,scope
str,i64,f64,f64,f64,str,str
"""precision_at_k""",10,0.732,0.647857,0.814286,"""bootstrap_posterior_simulation""","""internal_evaluation_only"""
"""recall_at_k""",10,0.740097,0.655215,0.835449,"""bootstrap_posterior_simulation""","""internal_evaluation_only"""
"""dollars_captured_at_k""",10,34134.0132,29988.238,39452.4105,"""bootstrap_posterior_simulation""","""internal_evaluation_only"""
"""dollar_capture_rate""",10,0.78855,0.716016,0.873669,"""bootstrap_posterior_simulation""","""internal_evaluation_only"""


**Review-budget interval plot:** This chart shows uncertainty around review-budget performance rather than a single point estimate. The methodology repeatedly resamples synthetic evaluation outcomes to show a plausible range for metrics such as queue precision, recall, and dollar capture under the same review budget.

In [5]:
(
    ggplot(intervals, aes("metric", "mean"))
    + geom_point()
    + ggtitle("Bayesian-Style Review Budget Intervals")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff51cbdd5b0>)

**Component superiority plot:** This chart compares which scoring component tends to perform better across internal synthetic regimes. It is useful for model governance because it shows whether rules, statistics, ML, exposure, or the hybrid score are consistently useful or only strong under certain synthetic conditions.

In [6]:
(
    ggplot(
        superiority,
        aes("left_signal", "right_signal", fill="win_probability"),
    )
    + geom_tile()
    + ggtitle("Pairwise Component Superiority")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff51cc007c0>)

**Effect-size interval plot:** This chart shows not just which component wins, but how large the performance difference appears to be. The methodology compares component metrics across scenario and seed units, then summarizes the uncertainty around those differences.

In [7]:
(
    ggplot(superiority, aes("left_signal", "mean_delta"))
    + geom_point(aes(size="samples", color="scenario"))
    + geom_errorbar(aes(ymin="lower_95", ymax="upper_95"), width=0.2)
    + ggtitle("Effect-Size Intervals")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff51cc74ad0>)

## Hierarchical Subgroup Diagnostics

Diagnostic question: where do raw subgroup anomaly rates differ from pooled estimates after targeted subgroup drift?

In [8]:
subgroups = subgroup_diagnostics(scored, k=10, scenario="subgroup-drift")
top_subgroups = top_subgroup_diagnostics(subgroups, top_n=15)
top_subgroups

scenario,dimension,subgroup,records,true_anomalies,reviewed_records,true_positive_reviews,false_negatives,false_positives,anomaly_count,raw_anomaly_rate,pooled_anomaly_rate,shrinkage,lower_95,upper_95,raw_pooled_delta
str,str,str,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64
"""subgroup-drift""","""shift_type""","""Double""",423,140,387,138,2,249,140,0.330969,0.313034,-0.017935,0.268842,0.357227,0.017935
"""subgroup-drift""","""pay_code_category""","""Overtime""",7653,128,797,126,2,671,128,0.016725,0.016702,-0.000023,0.013831,0.019573,0.000023
"""subgroup-drift""","""approval_status""","""Approved""",13354,120,722,118,2,604,120,0.008986,0.008987,0.000001,0.007386,0.010588,-0.000001
"""subgroup-drift""","""license_type""","""CNA""",6196,64,335,63,1,272,64,0.010329,0.010326,-0.000003,0.007809,0.012843,0.000003
"""subgroup-drift""","""role""","""CNA""",6196,64,335,63,1,272,64,0.010329,0.010326,-0.000003,0.007809,0.012843,0.000003
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""subgroup-drift""","""role""","""RN""",2273,27,160,27,0,133,27,0.011879,0.011853,-0.000025,0.007404,0.016303,0.000025
"""subgroup-drift""","""facility_id""","""SNF-F003""",2751,26,140,25,1,115,26,0.009451,0.009452,0.000001,0.005836,0.013068,-0.000001
"""subgroup-drift""","""facility_id""","""SNF-F001""",2597,25,140,25,0,115,25,0.009626,0.009626,-5.2585e-7,0.005871,0.013381,5.2585e-7


**Subgroup forest plot:** This chart highlights where synthetic anomaly-review outcomes differ across payroll subgroups such as departments. Stakeholders should use it as an internal diagnostic for concentration and coverage patterns, not as evidence about real employee groups.

In [9]:
department_subgroups = top_subgroups.filter(pl.col("dimension") == "department").sort(
    "pooled_anomaly_rate",
)
(
    ggplot(
        department_subgroups,
        aes("subgroup", "pooled_anomaly_rate"),
    )
    + geom_point(aes(size="records", color="scenario"))
    + geom_errorbar(aes(ymin="lower_95", ymax="upper_95"), width=0.2)
    + ggtitle("Subgroup Pooled Anomaly Rates")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff6a8389390>)

**Subgroup shrinkage plot:** This chart compares raw subgroup results with stabilized estimates that reduce overreaction to small groups. The methodology pulls sparse subgroup estimates toward the overall pattern so internal reviewers can distinguish stronger signals from noisy small-sample variation.

In [10]:
(
    ggplot(
        subgroups,
        aes("raw_anomaly_rate", "pooled_anomaly_rate"),
    )
    + geom_point(aes(size="records"))
    + ggtitle("Raw vs Pooled Subgroup Rates")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff51ccc31d0>)

## Expected-Pay And Exposure Calibration

Diagnostic question: are expected-pay intervals covering normal variation, and where do residuals or p90 excess concentrate?

In [11]:
calibration = calibration_plot_inputs(
    scored,
    scenario="subgroup-drift",
    by="department",
)
exposure = exposure_calibration(scored)
calibration

subgroup,records,coverage,avg_interval_width,avg_excess_over_p90,avg_residual,subgroup_dimension,scenario,interval_width,excess_over_p90,residual,tail_excess
str,u32,f64,f64,f64,f64,str,str,f64,f64,f64,f64
"""Facility Support""",3044,0.744342,61.160595,11.003802,12.27526,"""department""","""subgroup-drift""",61.160595,11.003802,12.27526,11.003802
"""Nursing""",11583,0.773815,95.39921,11.244069,10.990824,"""department""","""subgroup-drift""",95.39921,11.244069,10.990824,11.244069


**Expected-pay coverage plot:** This chart shows whether expected gross-pay intervals cover typical synthetic records across subgroups. It helps reviewers assess whether the expected-pay context is broad enough for normal variation without becoming too vague for triage.

In [12]:
(
    ggplot(calibration, aes("subgroup", "coverage"))
    + geom_point()
    + ggtitle("Expected Pay Coverage")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff51ccc3380>)

**Expected-pay residual plot:** This chart shows where actual gross pay sits relative to expected-pay estimates. Large residual patterns can indicate scenario drift, subgroup-specific pay behavior, or areas where expected-pay context may need recalibration before operational use.

In [13]:
(
    ggplot(calibration, aes("subgroup", "avg_residual"))
    + geom_point()
    + ggtitle("Expected Pay Residuals")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff51ccc2060>)

In [14]:
exposure

anomaly_category,records,avg_estimated_exposure,avg_synthetic_anomaly_dollars,total_estimated_exposure,total_synthetic_anomaly_dollars,exposure_to_synthetic_ratio
str,u32,f64,f64,f64,f64,f64
"""overtime_double_shift""",140,369.634796,311.464214,51748.8714,43604.99,1.186765
"""normal""",14487,36.282877,0.0,525630.04588,0.0,5.2563e14


## Robustness And Perturbation Sensitivity

Diagnostic question: which scenario/seed units are unstable enough to affect review queues?

In [15]:
alt_results = run_pipeline(
    PayrollConfig(
        employee_count=160,
        pay_periods=14,
        review_budgets=(10, 25),
        seed=config.seed + 1,
    ),
    include=active_pipeline_include,
)
robustness = robustness_summary(
    {
        "subgroup-drift|seed=42|origin=default": scored,
        "baseline|seed=43|origin=default": alt_results.scored,
    },
    k=10,
)
robustness

setting,scenario,seed,origin,k,precision_at_k,recall_at_k,f1_at_k,mean_performance,performance_variability,queue_size,mean_queue_overlap,queue_overlap,performance_instability,instability_metric
str,str,i64,str,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64
"""subgroup-drift|seed=42|origin=…","""subgroup-drift""",42,"""default""",10.0,0.757143,0.757143,0.757143,0.757143,0.0,10,0.0,0.0,1.0,1.0
"""baseline|seed=43|origin=defaul…","""baseline""",43,"""default""",10.0,0.664286,0.794872,0.723735,0.664286,0.0,10,0.0,0.0,1.0,1.0


**Instability Pareto plot:** This chart ranks internal scenario and seed units by instability so reviewers can focus on the settings most likely to change queue behavior. The methodology combines performance movement and queue-overlap changes into a diagnostic signal for robustness review.

In [16]:
(
    ggplot(
        robustness,
        aes("performance_instability", "precision_at_k"),
    )
    + geom_point()
    + ggtitle("Performance vs Instability")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff51ce231d0>)

In [17]:
def perturb_gross_pay(frame: pl.DataFrame) -> pl.DataFrame:
    return frame.with_columns((pl.col("gross_pay") * 1.02).alias("gross_pay"))


sensitivity = perturbation_sensitivity(
    scored,
    perturb_gross_pay,
    lambda frame: score_payroll(frame, config),
)
sensitivity.head(10)

record_id,final_anomaly_score,pay_period_rank,perturbed_score,perturbed_rank,score_movement,rank_movement,crossed_threshold
i64,f64,u32,f64,u32,f64,u32,bool
0,0.156613,49,0.161104,49,0.004492,0,false
1,0.071934,133,0.074384,136,0.00245,3,false
2,0.055787,151,0.056301,154,0.000514,3,false
3,0.069326,135,0.072706,138,0.00338,3,false
4,0.037627,165,0.034749,172,-0.002879,7,false
5,0.145013,55,0.15182,55,0.006807,0,false
78,0.116316,84,0.12393,83,0.007613,4294967295,false
79,0.137616,64,0.134768,71,-0.002849,7,false
80,0.209422,21,0.224546,20,0.015124,4294967295,false


**Perturbation sensitivity heatmap:** This chart shows which ranked records move most when a controlled input perturbation is applied. It helps internal reviewers see whether small synthetic input changes materially alter score or rank, which is important for trust in review prioritization.

In [18]:
(
    ggplot(sensitivity, aes("rank_movement", "score_movement"))
    + geom_point()
    + ggtitle("Perturbation Sensitivity")
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7ff51ce23800>)